# Chapter 4 Lab — Your First Agent

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 4 — Your First Agent  
**Goal:** Build a ReAct agent from scratch — no frameworks — and understand every piece of the observe-think-act loop.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | The Naive Agent (what NOT to do) |
| 2 | Building the ReAct Agent from scratch |
| 3 | Stopping Conditions |
| 4 | Error Recovery |
| 5 | Making It Configurable |
| 6 | Exercises |

> **Prerequisites:** An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install openai python-dotenv requests --quiet

In [ ]:
import json
import math
import os
import re
import time
import textwrap
from datetime import datetime
from typing import Callable

import openai
import requests
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

client = openai.OpenAI()

# Quick connectivity check
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'ready' in one word."}],
    max_tokens=5,
)
print("API check:", resp.choices[0].message.content)

---
## 1. The Naive Agent (What Not to Do)

The most common beginner mistake: give an LLM tools, call them **once**, and trust whatever comes back. This produces impressive demos but fails on any task that requires more than one reasoning step.

Below we ask a factual question that needs **two** lookups (Tokyo population + Japan population) and a calculation. The naive agent will hallucinate at least one number.

In [ ]:
def run_naive_agent(question: str) -> str:
    """One-shot: ask the LLM, execute every tool call once, return result.
    This is the WRONG way to build an agent — shown here for contrast."""

    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
        {"role": "user", "content": question},
    ]

    # Single LLM call — no loop, no reflection
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )

    return response.choices[0].message.content


# ── Test: ask a question that REQUIRES multiple lookups ──────────────────────
question = (
    "What is the current population of Tokyo, and what percentage "
    "of Japan's total population does that represent?"
)

naive_answer = run_naive_agent(question)
print("Naive agent answer:\n")
print(textwrap.fill(naive_answer, width=80))
print("\n⚠  The numbers above likely come from training data, not a live source.")
print("   The agent had no tools and no reasoning loop — it just guessed.")

In [ ]:
# ── More hallucination examples ──────────────────────────────────────────────
# Try questions where stale or invented facts are easy to spot.

hallucination_questions = [
    "Who is the current CEO of OpenAI and when did they start?",
    "What was the closing price of AAPL stock last Friday?",
    "How many days until the next solar eclipse visible from North America?",
]

for q in hallucination_questions:
    ans = run_naive_agent(q)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

print("--- None of these answers can be verified. The agent has no tools. ---")

---
## 2. Building the ReAct Agent

The fix is structural: instead of "call tools once and assemble," we build a **loop** where the agent **Observes** (reads tool results), **Thinks** (reasons about what to do next), and **Acts** (calls a tool or emits a final answer) — repeating until the task is genuinely complete.

This is the **ReAct** pattern (Yao et al., 2022).

### 2.1 Tool Registry with Decorator Pattern

In [ ]:
# ── Tool Registry ─────────────────────────────────────────────────────────────
# A tool is a Python function + a JSON schema the LLM can read.
# The @tool decorator registers both in a global dict.

TOOL_REGISTRY: dict[str, tuple[Callable, dict]] = {}


def tool(name: str, description: str, parameters: dict):
    """Decorator that registers a function as an agent tool."""
    def decorator(func: Callable) -> Callable:
        schema = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": {
                    "type": "object",
                    "properties": parameters,
                    "required": list(parameters.keys()),
                },
            },
        }
        TOOL_REGISTRY[name] = (func, schema)
        return func
    return decorator


def get_tool_schemas() -> list[dict]:
    """Return OpenAI-compatible tool schemas for all registered tools."""
    return [schema for _, schema in TOOL_REGISTRY.values()]


def execute_tool(name: str, arguments: dict) -> str:
    """Look up a tool by name and execute it with the given arguments."""
    if name not in TOOL_REGISTRY:
        return f"Error: Unknown tool '{name}'. Available: {list(TOOL_REGISTRY.keys())}"
    func, _ = TOOL_REGISTRY[name]
    try:
        return func(**arguments)
    except Exception as e:
        return f"Error executing {name}: {e}"


print(f"Registry ready. {len(TOOL_REGISTRY)} tools registered so far.")

### 2.2 Define Three Tools

| Tool | What it does |
|------|-------------|
| `search_wikipedia` | Queries the Wikipedia API — returns a real text snippet |
| `calculator` | Evaluates a math expression safely (no `exec`) |
| `get_current_date` | Returns today's date and time |

In [ ]:
# ── Tool 1: Wikipedia Search ──────────────────────────────────────────────────

@tool(
    name="search_wikipedia",
    description=(
        "Search Wikipedia for factual information. "
        "Returns a text snippet from the best-matching article."
    ),
    parameters={
        "query": {
            "type": "string",
            "description": "The search query, e.g. 'population of Tokyo'",
        }
    },
)
def search_wikipedia(query: str) -> str:
    """Hit the Wikipedia REST API and return the first extract."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": 1,
        "format": "json",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        results = resp.json().get("query", {}).get("search", [])
        if not results:
            return f"No Wikipedia results for '{query}'."

        # Fetch the extract for the top hit
        title = results[0]["title"]
        extract_params = {
            "action": "query",
            "titles": title,
            "prop": "extracts",
            "exintro": True,
            "explaintext": True,
            "format": "json",
        }
        ext_resp = requests.get(url, params=extract_params, timeout=10)
        ext_resp.raise_for_status()
        pages = ext_resp.json().get("query", {}).get("pages", {})
        for page in pages.values():
            extract = page.get("extract", "")
            if extract:
                # Truncate to keep context window manageable
                return f"Wikipedia — {title}:\n{extract[:1500]}"
        return f"Found article '{title}' but no extract available."
    except requests.RequestException as e:
        return f"Error searching Wikipedia: {e}"


# Quick test
print(search_wikipedia("population of Tokyo")[:300], "...")

In [ ]:
# ── Tool 2: Calculator ────────────────────────────────────────────────────────

@tool(
    name="calculator",
    description="Evaluate a mathematical expression. Returns the numeric result as a string.",
    parameters={
        "expression": {
            "type": "string",
            "description": "A valid Python math expression, e.g. '14.0 / 123.9 * 100'",
        }
    },
)
def calculator(expression: str) -> str:
    """Safely evaluate a math expression (no exec, no imports)."""
    allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)  # noqa: S307
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# Quick test
print("2 + 2 =", calculator("2 + 2"))
print("sqrt(144) =", calculator("sqrt(144)"))
print("bad expr =", calculator("import os"))  # should fail safely

In [ ]:
# ── Tool 3: Get Current Date ──────────────────────────────────────────────────

@tool(
    name="get_current_date",
    description="Return the current date and time in ISO format. Takes no arguments.",
    parameters={},
)
def get_current_date() -> str:
    """Return the current UTC date/time."""
    return datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


# Quick test
print("Current date:", get_current_date())
print(f"\nAll registered tools: {list(TOOL_REGISTRY.keys())}")

### 2.3 The ReAct Prompt Template

The system prompt is the **contract** the loop depends on. It tells the model:
1. Always think before acting (prefix with `Thought:`)
2. Signal completion with `FINAL ANSWER:`
3. Never guess when you can look things up

In [ ]:
SYSTEM_PROMPT = """You are a reasoning agent. You solve tasks by thinking
step-by-step and using tools when needed.

For EVERY step, you MUST output your reasoning in a "Thought:" prefix before
deciding to call a tool or give a final answer.

Rules:
1. Always think before acting. Never call a tool without explaining why.
2. After receiving a tool result, think about what it means before continuing.
3. When you have enough information, respond with "FINAL ANSWER:" followed
   by your complete answer.
4. If a tool returns an error, think about why and try a different approach.
5. Never guess when you can look things up.
"""

print(SYSTEM_PROMPT)

### 2.4 The Agent Loop

The core class: observe (context window) -> think (LLM generates thought) -> act (tool call or final answer) -> repeat.

In [ ]:
class ReActAgent:
    """A from-scratch ReAct agent — no frameworks required."""

    def __init__(
        self,
        model: str = "gpt-4o",
        max_iterations: int = 10,
        verbose: bool = True,
    ):
        self.model = model
        self.max_iterations = max_iterations
        self.verbose = verbose
        self.client = openai.OpenAI()

    def run(self, task: str) -> str:
        """Execute the ReAct loop until FINAL ANSWER or max iterations."""
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task},
        ]
        tool_schemas = get_tool_schemas()

        for i in range(self.max_iterations):
            if self.verbose:
                print(f"\n{'='*60}")
                print(f"--- Iteration {i + 1} ---")

            # THINK: ask the LLM for a thought + optional tool call
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tool_schemas if tool_schemas else None,
                tool_choice="auto",
            )
            message = response.choices[0].message
            messages.append(message)

            # Check for text content (the "Thought" part)
            if message.content:
                if self.verbose:
                    print(f"Thought: {message.content}")

                # Check for final answer
                if "FINAL ANSWER:" in message.content:
                    answer = message.content.split("FINAL ANSWER:")[-1].strip()
                    if self.verbose:
                        print(f"\n{'='*60}")
                        print(f"DONE in {i + 1} iteration(s).")
                    return answer

            # ACT: execute any tool calls
            if message.tool_calls:
                for tool_call in message.tool_calls:
                    name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)

                    if self.verbose:
                        print(f"Action: {name}({json.dumps(args)})")

                    result = execute_tool(name, args)

                    if self.verbose:
                        print(f"Observation: {result[:300]}")

                    # OBSERVE: feed the result back
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result,
                    })

            elif not message.content:
                # No content and no tool calls — nudge the model
                messages.append({
                    "role": "user",
                    "content": "Please continue reasoning about the task.",
                })

        # Max iterations reached — force a final answer
        return self._force_final_answer(messages)

    def _force_final_answer(self, messages: list) -> str:
        """Force the agent to produce a final answer after max iterations."""
        messages.append({
            "role": "user",
            "content": (
                "You have reached the maximum number of reasoning steps. "
                "Based on everything you have gathered so far, provide your "
                "best FINAL ANSWER: now."
            ),
        })
        response = self.client.chat.completions.create(
            model=self.model, messages=messages,
        )
        content = response.choices[0].message.content or ""
        if "FINAL ANSWER:" in content:
            return content.split("FINAL ANSWER:")[-1].strip()
        return content


print("ReActAgent class defined.")

### 2.5 Run the Agent — Full Reasoning Traces

Let's run the agent on several questions and watch the full Thought/Action/Observation trace. Compare the quality to the naive agent above.

In [ ]:
# ── Test 1: The same Tokyo question that tripped up the naive agent ───────────
agent = ReActAgent(model="gpt-4o-mini", max_iterations=10, verbose=True)

answer = agent.run(
    "What is the current population of Tokyo, and what percentage "
    "of Japan's total population does that represent?"
)
print(f"\n{'='*60}")
print(f"FINAL ANSWER: {answer}")

In [ ]:
# ── Test 2: Multi-step with date + calculation ───────────────────────────────
answer2 = agent.run(
    "What is today's date, and how many days are left until the end of the year? "
    "Use the calculator to compute it precisely."
)
print(f"\n{'='*60}")
print(f"FINAL ANSWER: {answer2}")

In [ ]:
# ── Test 3: Research + synthesis ──────────────────────────────────────────────
answer3 = agent.run(
    "Who wrote the novel '1984', and what year was it published? "
    "Also, how many years ago was that from today's date?"
)
print(f"\n{'='*60}")
print(f"FINAL ANSWER: {answer3}")

In [ ]:
# ── Test 4: A question requiring multiple Wikipedia lookups ──────────────────
answer4 = agent.run(
    "Compare the land area of France and Germany. Which is larger, "
    "and by how many square kilometers?"
)
print(f"\n{'='*60}")
print(f"FINAL ANSWER: {answer4}")

---
## 3. Stopping Conditions

An agent must know when to stop. A well-designed agent uses **four** stopping conditions:

| # | Condition | What it catches |
|---|-----------|----------------|
| 1 | **Explicit completion** | Agent says `FINAL ANSWER:` |
| 2 | **Maximum iterations** | Hard cap prevents infinite loops |
| 3 | **Stuck detection** | Same action repeated 3 times |
| 4 | **Token budget** | Cumulative tokens exceed a threshold |

Below we build an enhanced agent that implements all four.

In [ ]:
class ReActAgentV2:
    """ReAct agent with all four stopping conditions."""

    def __init__(
        self,
        model: str = "gpt-4o",
        max_iterations: int = 10,
        stuck_window: int = 3,
        token_budget: int = 50_000,
        verbose: bool = True,
    ):
        self.model = model
        self.max_iterations = max_iterations
        self.stuck_window = stuck_window
        self.token_budget = token_budget
        self.verbose = verbose
        self.client = openai.OpenAI()
        # Tracking
        self.total_tokens = 0
        self.stop_reason = None

    # ── Stopping condition 3: Stuck detection ────────────────────────────────
    def _is_stuck(self, messages: list) -> bool:
        """Return True if the last `stuck_window` tool calls are identical."""
        recent_calls = []
        for msg in reversed(messages):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    recent_calls.append(
                        (tc.function.name, tc.function.arguments)
                    )
                    if len(recent_calls) >= self.stuck_window:
                        return len(set(recent_calls)) == 1
        return False

    # ── Stopping condition 4: Token budget ───────────────────────────────────
    def _estimate_tokens(self, messages: list) -> int:
        """Rough estimate: ~4 chars per token."""
        return sum(len(str(m)) for m in messages) // 4

    def run(self, task: str) -> dict:
        """Run the agent. Returns {answer, stop_reason, total_tokens, iterations}."""
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task},
        ]
        tool_schemas = get_tool_schemas()
        self.total_tokens = 0

        for i in range(self.max_iterations):
            # ── Check stopping condition 3: stuck ────────────────────────
            if self._is_stuck(messages):
                if self.verbose:
                    print(f"\n[!] Stuck detected at iteration {i+1} — "
                          f"same action repeated {self.stuck_window} times.")
                messages.append({
                    "role": "user",
                    "content": (
                        "You are repeating the same action. "
                        "Try a completely different approach, or if you "
                        "have enough information, provide your FINAL ANSWER:."
                    ),
                })

            # ── Check stopping condition 4: token budget ─────────────────
            estimated = self._estimate_tokens(messages)
            if estimated > self.token_budget:
                if self.verbose:
                    print(f"\n[!] Token budget exceeded: ~{estimated} > {self.token_budget}")
                self.stop_reason = "token_budget"
                answer = self._force_final_answer(messages)
                return self._result(answer, i + 1)

            if self.verbose:
                print(f"\n{'='*60}")
                print(f"--- Iteration {i+1} | ~{estimated} tokens used ---")

            # ── THINK ────────────────────────────────────────────────────
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tool_schemas if tool_schemas else None,
                tool_choice="auto",
            )
            # Track real token usage from the API
            if response.usage:
                self.total_tokens += response.usage.total_tokens

            message = response.choices[0].message
            messages.append(message)

            # ── Check stopping condition 1: explicit completion ──────────
            if message.content:
                if self.verbose:
                    print(f"Thought: {message.content[:500]}")
                if "FINAL ANSWER:" in message.content:
                    self.stop_reason = "explicit_completion"
                    answer = message.content.split("FINAL ANSWER:")[-1].strip()
                    return self._result(answer, i + 1)

            # ── ACT ─────────────────────────────────────────────────────
            if message.tool_calls:
                for tc in message.tool_calls:
                    name = tc.function.name
                    args = json.loads(tc.function.arguments)
                    if self.verbose:
                        print(f"Action: {name}({json.dumps(args)})")
                    result = execute_tool(name, args)
                    if self.verbose:
                        print(f"Observation: {result[:300]}")
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result,
                    })
            elif not message.content:
                messages.append({
                    "role": "user",
                    "content": "Please continue reasoning about the task.",
                })

        # ── Stopping condition 2: max iterations ────────────────────────────
        if self.verbose:
            print(f"\n[!] Max iterations ({self.max_iterations}) reached.")
        self.stop_reason = "max_iterations"
        answer = self._force_final_answer(messages)
        return self._result(answer, self.max_iterations)

    def _force_final_answer(self, messages: list) -> str:
        messages.append({
            "role": "user",
            "content": "Maximum steps reached. Provide your FINAL ANSWER: now.",
        })
        response = self.client.chat.completions.create(
            model=self.model, messages=messages,
        )
        if response.usage:
            self.total_tokens += response.usage.total_tokens
        content = response.choices[0].message.content or "Unable to determine."
        if "FINAL ANSWER:" in content:
            return content.split("FINAL ANSWER:")[-1].strip()
        return content

    def _result(self, answer: str, iterations: int) -> dict:
        return {
            "answer": answer,
            "stop_reason": self.stop_reason,
            "total_tokens": self.total_tokens,
            "iterations": iterations,
        }


print("ReActAgentV2 defined with all 4 stopping conditions.")

In [ ]:
# ── Test: Explicit completion (happy path) ───────────────────────────────────
agent_v2 = ReActAgentV2(model="gpt-4o-mini", max_iterations=10, verbose=True)

result = agent_v2.run("What is the capital of France?")
print(f"\nResult: {json.dumps(result, indent=2)}")
assert result["stop_reason"] == "explicit_completion", f"Expected explicit_completion, got {result['stop_reason']}"
print("Stop reason: explicit_completion (as expected)")

In [ ]:
# ── Test: Maximum iterations (cap at 3 to trigger it quickly) ────────────────
agent_limited = ReActAgentV2(
    model="gpt-4o-mini", max_iterations=3, verbose=True
)

result_limited = agent_limited.run(
    "List the GDP of every country in the G20. This requires many lookups."
)
print(f"\nResult: {json.dumps(result_limited, indent=2)}")
print(f"Stop reason: {result_limited['stop_reason']}")

In [ ]:
# ── Test: Token budget (set very low to trigger it) ──────────────────────────
agent_budget = ReActAgentV2(
    model="gpt-4o-mini",
    max_iterations=10,
    token_budget=500,   # Very low — will trigger after 1-2 iterations
    verbose=True,
)

result_budget = agent_budget.run(
    "Search Wikipedia for the history of the Roman Empire and summarize it."
)
print(f"\nResult: {json.dumps(result_budget, indent=2)}")
print(f"Stop reason: {result_budget['stop_reason']}")

---
## 4. Error Recovery

Tools fail. APIs time out. Search queries return nothing useful. A robust agent handles these gracefully instead of crashing or hallucinating.

**Strategy:** Treat errors as observations. The agent sees the error in the next reasoning step and can adapt.

Below we deliberately break the Wikipedia tool to simulate a timeout, then watch the agent recover.

In [ ]:
# ── Simulate tool failure ─────────────────────────────────────────────────────
# We monkey-patch search_wikipedia to fail the first 2 calls,
# then succeed on the 3rd. This simulates a flaky API.

_original_search = search_wikipedia.__wrapped__ if hasattr(search_wikipedia, '__wrapped__') else None

_call_count = 0
_original_fn, _original_schema = TOOL_REGISTRY["search_wikipedia"]


def flaky_search_wikipedia(query: str) -> str:
    """Wrapper that fails the first 2 calls, then delegates to the real tool."""
    global _call_count
    _call_count += 1
    if _call_count <= 2:
        raise TimeoutError(
            f"Connection timed out after 10s (attempt {_call_count}). "
            "The Wikipedia API appears to be down."
        )
    # On 3rd+ call, use the real function
    return _original_fn(query)


# Swap the function in the registry (keep the same schema)
TOOL_REGISTRY["search_wikipedia"] = (flaky_search_wikipedia, _original_schema)

print("search_wikipedia is now FLAKY (fails first 2 calls, then works).")
print("Let's see how the agent handles it...\n")

# ── Run the agent against the flaky tool ─────────────────────────────────────
_call_count = 0  # reset counter
agent_err = ReActAgentV2(model="gpt-4o-mini", max_iterations=10, verbose=True)

result_err = agent_err.run(
    "What is the population of Berlin according to Wikipedia?"
)

print(f"\n{'='*60}")
print(f"Answer: {result_err['answer']}")
print(f"Stop reason: {result_err['stop_reason']}")
print(f"Iterations used: {result_err['iterations']}")
print(f"\nThe agent recovered from 2 timeouts and still found the answer.")

In [ ]:
# ── Restore the real tool ─────────────────────────────────────────────────────
TOOL_REGISTRY["search_wikipedia"] = (_original_fn, _original_schema)
print("search_wikipedia restored to normal operation.")

---
## 5. Making It Configurable

A useful agent is a **configurable** agent. The version below accepts tools, prompts, and settings as configuration — so you can build different agents from the same class by swapping what gets plugged in.

Key changes from v2:
- Tools passed in as a dictionary (not pulled from a global registry)
- System prompt is customizable
- `run()` returns a structured trace alongside the answer

In [ ]:
class ConfigurableReActAgent:
    """A fully configurable ReAct agent with pluggable tools and prompts."""

    def __init__(
        self,
        tools: dict[str, tuple[Callable, dict]] | None = None,
        model: str = "gpt-4o",
        system_prompt: str | None = None,
        max_iterations: int = 10,
        stuck_window: int = 3,
        token_budget: int = 50_000,
        verbose: bool = True,
    ):
        self.tools = tools or {}
        self.model = model
        self.max_iterations = max_iterations
        self.stuck_window = stuck_window
        self.token_budget = token_budget
        self.verbose = verbose
        self.client = openai.OpenAI()

        # Build the system prompt — auto-list available tools
        tool_descriptions = "\n".join(
            f"- {name}: {schema['function']['description']}"
            for name, (_, schema) in self.tools.items()
        )
        self.system_prompt = system_prompt or (
            "You are a reasoning agent. Think step-by-step.\n\n"
            f"Available tools:\n{tool_descriptions}\n\n"
            "Rules:\n"
            "1. Always explain your reasoning before calling a tool.\n"
            "2. When done, respond with FINAL ANSWER: followed by your answer.\n"
            "3. If a tool fails, reason about why and try a different approach.\n"
            "4. Never guess when you can look things up.\n"
        )

    def add_tool(self, name: str, func: Callable, schema: dict):
        """Register a new tool after initialization."""
        self.tools[name] = (func, schema)

    def _get_tool_schemas(self) -> list[dict]:
        return [schema for _, schema in self.tools.values()]

    def _execute_tool(self, name: str, arguments: dict) -> str:
        if name not in self.tools:
            return f"Error: Unknown tool '{name}'. Available: {list(self.tools.keys())}"
        func, _ = self.tools[name]
        try:
            return func(**arguments)
        except Exception as e:
            return f"Error executing {name}: {e}"

    def _is_stuck(self, messages: list) -> bool:
        recent = []
        for msg in reversed(messages):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    recent.append((tc.function.name, tc.function.arguments))
                    if len(recent) >= self.stuck_window:
                        return len(set(recent)) == 1
        return False

    def _estimate_tokens(self, messages: list) -> int:
        return sum(len(str(m)) for m in messages) // 4

    def run(self, task: str) -> dict:
        """Run the agent. Returns {answer, trace, stop_reason, total_tokens, iterations}."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": task},
        ]
        schemas = self._get_tool_schemas()
        trace = []
        total_tokens = 0

        for i in range(self.max_iterations):
            if self._is_stuck(messages):
                messages.append({
                    "role": "user",
                    "content": "You are repeating yourself. Try a different approach or FINAL ANSWER:.",
                })
                trace.append({"type": "system", "content": "Stuck detection triggered"})

            estimated = self._estimate_tokens(messages)
            if estimated > self.token_budget:
                answer = self._force_final(messages)
                return {"answer": answer, "trace": trace, "stop_reason": "token_budget",
                        "total_tokens": total_tokens, "iterations": i + 1}

            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=schemas or None,
                tool_choice="auto",
            )
            if response.usage:
                total_tokens += response.usage.total_tokens

            msg = response.choices[0].message
            messages.append(msg)

            if msg.content:
                trace.append({"type": "thought", "content": msg.content})
                if self.verbose:
                    print(f"[Step {i+1}] Thought: {msg.content[:300]}")
                if "FINAL ANSWER:" in msg.content:
                    answer = msg.content.split("FINAL ANSWER:")[-1].strip()
                    return {"answer": answer, "trace": trace, "stop_reason": "explicit_completion",
                            "total_tokens": total_tokens, "iterations": i + 1}

            if msg.tool_calls:
                for tc in msg.tool_calls:
                    name = tc.function.name
                    args = json.loads(tc.function.arguments)
                    trace.append({"type": "action", "tool": name, "args": args})
                    result = self._execute_tool(name, args)
                    trace.append({"type": "observation", "content": result})
                    if self.verbose:
                        print(f"[Step {i+1}] Action: {name}({json.dumps(args)})")
                        print(f"[Step {i+1}] Result: {result[:200]}")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        answer = self._force_final(messages)
        return {"answer": answer, "trace": trace, "stop_reason": "max_iterations",
                "total_tokens": total_tokens, "iterations": self.max_iterations}

    def _force_final(self, messages: list) -> str:
        messages.append({"role": "user", "content": "Maximum steps reached. FINAL ANSWER: now."})
        resp = self.client.chat.completions.create(model=self.model, messages=messages)
        content = resp.choices[0].message.content or ""
        if "FINAL ANSWER:" in content:
            return content.split("FINAL ANSWER:")[-1].strip()
        return content


print("ConfigurableReActAgent defined.")

### 5.1 Run with different tool sets

The same agent class, two different configurations:
- **Config A:** Full toolkit (Wikipedia + calculator + date)
- **Config B:** Calculator only (no search — the agent must reason from what it knows)

In [ ]:
# ── Helper: build a tool dict entry from the global registry ─────────────────
def pick_tools(*names: str) -> dict[str, tuple[Callable, dict]]:
    """Select a subset of tools from the global TOOL_REGISTRY."""
    return {name: TOOL_REGISTRY[name] for name in names if name in TOOL_REGISTRY}


# ── Config A: Full toolkit ───────────────────────────────────────────────────
agent_full = ConfigurableReActAgent(
    tools=pick_tools("search_wikipedia", "calculator", "get_current_date"),
    model="gpt-4o-mini",
    verbose=True,
)

question = "How old was Albert Einstein when he published his theory of general relativity?"

print("=" * 60)
print("CONFIG A: Full toolkit (search + calculator + date)")
print("=" * 60)
result_a = agent_full.run(question)
print(f"\nAnswer: {result_a['answer']}")
print(f"Tools used: {[s['tool'] for s in result_a['trace'] if s['type'] == 'action']}")

In [ ]:
# ── Config B: Calculator only ─────────────────────────────────────────────────
agent_calc_only = ConfigurableReActAgent(
    tools=pick_tools("calculator"),
    model="gpt-4o-mini",
    verbose=True,
)

print("=" * 60)
print("CONFIG B: Calculator only (no search)")
print("=" * 60)
result_b = agent_calc_only.run(question)
print(f"\nAnswer: {result_b['answer']}")
print(f"Tools used: {[s['tool'] for s in result_b['trace'] if s['type'] == 'action']}")

print("\n" + "=" * 60)
print("Notice: Config B had to rely on training data for dates (may be wrong).")
print("Config A could verify facts via Wikipedia.")

In [ ]:
# ── Inspect the full trace from Config A ─────────────────────────────────────
print("Full reasoning trace (Config A):\n")
for i, step in enumerate(result_a["trace"]):
    step_type = step["type"].upper()
    if step_type == "ACTION":
        print(f"  [{i}] {step_type}: {step['tool']}({json.dumps(step['args'])})")
    elif step_type == "OBSERVATION":
        print(f"  [{i}] {step_type}: {step['content'][:120]}...")
    else:
        print(f"  [{i}] {step_type}: {step['content'][:120]}...")

print(f"\nTotal tokens: {result_a['total_tokens']}")
print(f"Iterations: {result_a['iterations']}")

---
## 6. Exercises

### Exercise 1 — Trace Analysis (Conceptual)

Look at the trace printed below. Identify:
1. Which step contained the **planning** decision?
2. Where did the agent **ground** its reasoning in tool output (instead of guessing)?
3. If you removed the Thought steps, what would go wrong?

Write your answers in the markdown cell that follows the code cell.

In [ ]:
# ── Exercise 1: Run a multi-step question, then analyze the trace ────────────
exercise_agent = ConfigurableReActAgent(
    tools=pick_tools("search_wikipedia", "calculator", "get_current_date"),
    model="gpt-4o-mini",
    verbose=True,
)

exercise_result = exercise_agent.run(
    "What is the speed of light in km/h? Search for the value in m/s, "
    "then convert it using the calculator."
)

print("\n\nFull trace for analysis:")
for i, step in enumerate(exercise_result["trace"]):
    print(f"  Step {i}: [{step['type']}] ", end="")
    if step["type"] == "action":
        print(f"{step['tool']}({step['args']})")
    else:
        print(step["content"][:150])

**Your answers here:**

1. Planning step: *...*
2. Grounding step: *...*
3. Without Thought steps: *...*

---

### Exercise 2 — Add a New Tool (Coding)

Add a `string_length` tool that counts the number of characters in a given string. Then run the configurable agent with this new tool to answer: *"How many characters are in the full name 'Johann Sebastian Bach'?"*

In [ ]:
# ── Exercise 2: YOUR CODE HERE ───────────────────────────────────────────────
# Step 1: Define the tool function and schema

def string_length(text: str) -> str:
    """Count the number of characters in a string."""
    # TODO: implement this
    pass

string_length_schema = {
    "type": "function",
    "function": {
        "name": "string_length",
        "description": "Count the number of characters in a given string.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The string to measure",
                }
            },
            "required": ["text"],
        },
    },
}

# Step 2: Create the agent with the new tool + calculator
# exercise_tools = {
#     "string_length": (string_length, string_length_schema),
#     "calculator": TOOL_REGISTRY["calculator"],
# }
# exercise2_agent = ConfigurableReActAgent(tools=exercise_tools, model="gpt-4o-mini")

# Step 3: Run it
# result = exercise2_agent.run("How many characters are in 'Johann Sebastian Bach'?")
# print(result["answer"])

### Exercise 3 — Multi-Domain Agent Config (Design)

Design two agent configurations for different domains. For each, specify:
- Which tools it needs (at least 2 per config)
- A custom system prompt tailored to the domain
- One test question

Implement at least one of them and run it.

**Example domains:** healthcare (PubMed search + drug interaction checker), finance (stock lookup + calculator), education (curriculum search + quiz generator).

In [ ]:
# ── Exercise 3: YOUR CODE HERE ───────────────────────────────────────────────
# Design and implement a domain-specific agent configuration.
#
# Example skeleton for a "Science Tutor" agent:
#
# science_prompt = """You are a science tutor agent. You help students
# understand scientific concepts by looking up facts and performing
# calculations. Always cite your sources (Wikipedia articles).
# When done, respond with FINAL ANSWER:."""
#
# science_agent = ConfigurableReActAgent(
#     tools=pick_tools("search_wikipedia", "calculator"),
#     model="gpt-4o-mini",
#     system_prompt=science_prompt,
#     verbose=True,
# )
#
# result = science_agent.run(
#     "What is the escape velocity of Earth in km/s? "
#     "Look up the formula and Earth's mass/radius, then calculate it."
# )
# print(result["answer"])

print("Uncomment and modify the code above, or write your own domain agent.")

---
## Key Takeaways

1. **A naive agent is not an agent** — it is a pipeline. Without a reasoning loop, the LLM calls tools once, trusts whatever comes back, and hallucates missing data.

2. **ReAct = Reasoning + Acting.** Alternating between free-form thoughts and structured tool calls produces dramatically better results. The thought step forces planning; the action step grounds the agent in real data.

3. **Four stopping conditions** prevent runaway agents: explicit completion, iteration limit, stuck detection, and token budget. Use all four in production.

4. **Errors are just observations.** The ReAct loop naturally handles tool failures — the agent sees the error and reasons about how to recover.

5. **Configuration over code.** A well-designed agent accepts tools, prompts, and settings as parameters. The same agent class can serve wildly different domains by changing what gets plugged in.